**Lab 4**

Use word embeddings to improve prompts for Generative AI model. Retrieve similar words using word embeddings. Use the similar words to enrich a Gen AI prompt. Use the AI model to generate responses for the original and enriched prompts. Compare the outputs in terms of detail and relevance.

In [1]:
import nltk, string
import gensim.downloader as ptwv

from transformers import pipeline
from nltk.tokenize import word_tokenize

/home/bti/BAIL657AI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/bti/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/bti/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [12]:
print("Loading pre-trained word vectors...")
word_vectors = ptwv.load("glove-wiki-gigaword-100")

Loading pre-trained word vectors...


In [28]:
def replace_keyword_in_prompt(prompt, keyword, word_vectors, topn=1):
    words = word_tokenize(prompt)
    enriched_words = []

    for word in words:
        cleaned_word = word.lower().strip(string.punctuation)
        if cleaned_word == keyword.lower():
            try:
                similar_words = word_vectors.most_similar(cleaned_word, topn=topn)

                if similar_words:
                    replacement_word = similar_words[0][0]
                    print(f"Replacing '{word}' -> '{replacement_word}'")
                    enriched_words.append(replacement_word)
                    continue
            except KeyError:
                print(f"'{keyword}' not found in the vocabulary, using original word.")
                
        enriched_words.append(word)
        
    enriched_prompt = " ".join(enriched_words)
    print(f"\nEnriched Prompt: {enriched_prompt}")
    return enriched_prompt

In [29]:
print("Loading GPT-2 model...")
generator = pipeline("text-generation", model="gpt2")


def generate_response(prompt, max_length=100):
    try:
        response = generator(
            prompt,
            max_length=max_length,
            num_return_sequences=1,
            truncation=True,
            pad_token_id=50256,
        )
        return response[0]["generated_text"]
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

Loading GPT-2 model...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1944.36it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
original_prompt = "Who is king?"
print(f"Original Prompt: {original_prompt}")

key_term = "king"
enriched_prompt = replace_keyword_in_prompt(original_prompt, key_term, word_vectors)

print("\nGenerating response for the original prompt...")
original_response = generate_response(original_prompt)

print(f"\nOriginal Prompt Response:\n{original_response}")

print("\nGenerating response for the enriched prompt...")
enriched_response = generate_response(enriched_prompt)

print(f"\nEnriched Prompt Response:\n{enriched_response}")

print("\nComparison of Responses:")
print(f"Original Prompt Response Length: {len(original_response)}")
print(f"Enriched Prompt Response Length: {len(enriched_response)}")
print(f"Original Prompt Response Detail: {original_response.count(".")}")
print(f"Enriched Prompt Response Detail: {enriched_response.count(".")}")

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original Prompt: Who is king?
Replacing 'king' -> 'prince'

Enriched Prompt: Who is prince ?

Generating response for the original prompt...


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original Prompt Response:
Who is king? Does he have a son? Is he the son of Ham? And is he the son of Baratheon? And is he the son of David? And is he the son of Haggai? And is he the son of Ahab? And is he the son of Ananias? And is he the son of A.H.? When is the child the son of David? Is he the son of Ham? Is he the son of David? And is he the son of A.H.?

And shall it come to pass that the son of David is the son of Ham? And shall the son of Ham be the son of Ahab? And shall the son of David be the son of Ahab? And shall the son of David be the son of Ahab? And shall the son of David be the son of Ahab? And shall the son of David be the son of Ahab? And shall the son of David be the son of Ahab? And shall the son of David be the son of Ahab?

and the father of Abraham shall be the son of Ham? And shall the son of Ham be the son of Ahab? And shall the son of David be the son of Ahab? And shall the

Generating response for the enriched prompt...

Enriched Prompt Response:
Who is p